# 02 ? flat training

**Objective:** Configure, launch and monitor one validation-selected EfficientNet-B0 reproduction run.

**Experiment:** Historical Flat versus Shared-Hard baseline reconstruction, seed 42.

**Config:** `configs/experiments/flat/efficientnet_b0.yaml`

**Inputs:** Config, verified train/validation images; last checkpoint only when explicitly resuming.

**Outputs:** Run snapshot/environment/history/logs, immutable best.pt and last.pt generations, registry events.

**Mode:** Training workflow; opt-in disabled by default. Every input is loaded from disk; no other notebook's kernel state is required.

Use **Restart Kernel ? Run All**. Real training is disabled until the build handover is reviewed.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/protocol.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from src.config import load_config
DATA_ROOT = os.environ.get("SKIN_CANCER_DATA_ROOT")
print("Dataset root:", Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else "NOT CONFIGURED ? set SKIN_CANCER_DATA_ROOT before image verification/training")


In [ ]:
CONFIG_PATH = "configs/experiments/flat/efficientnet_b0.yaml"
config = load_config(CONFIG_PATH)
display(config)

## Launch controls

After explicit Azure training authorization, set `RUN_TRAINING=True`. Use `RESUME=True` only for an interrupted run. A completed run cannot be overwritten. Other backbones remain locked until the EfficientNet-B0 comparison is reviewed.

In [ ]:
RUN_TRAINING = False
RESUME = False
DEVICE = "cuda"
from src.training import train
result = train(config, data_root=DATA_ROOT, allow_training=RUN_TRAINING, resume=RESUME, device=DEVICE)
display(result)

## Monitor and inspect checkpoint references

Re-run this cell to reload the current disk history. Validation selects the model; no internal-test predictions enter training.

In [ ]:
from src.reporting import run_status, plot_history
import json
summary, history = run_status(config["experiment_id"])
display(summary)
display(history.tail(10))
figure = plot_history(history)
if figure is not None: display(figure)
refs = ROOT/"experiments/runs"/config["experiment_id"]/"checkpoints.json"
if refs.exists(): display(json.loads(refs.read_text()))
else: print("Checkpoint artifacts: NOT CREATED")

## Summary and next step

Review the status and metrics displayed above. Missing artifacts mean **not run**, never a successful reproduction. Generated artifacts are listed in the output cells; scientific results remain separate from historical reference values.

**Next:** `03_shared_hierarchical_training.ipynb`. Preserve completed run directories, prediction files and checkpoint backups before continuing.
